In [1]:
!pip install imbalanced-learn

## Import Libraries

In [2]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.utils.class_weight import compute_class_weight

from imblearn.over_sampling import SMOTE

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.utils import to_categorical

2026-03-15 09:22:54.215047: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773566574.456302      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773566574.521278      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773566575.106221      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773566575.106285      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773566575.106288      55 computation_placer.cc:177] computation placer alr

In [12]:
dataset_path = "/kaggle/input/datasets/subhajournal/busi-breast-ultrasound-images-dataset/Dataset_BUSI_with_GT"

classes = ["benign", "malignant", "normal"]

labels = {"benign":0, "malignant":1, "normal":2}

## Load Image

In [13]:
X = []
y = []

img_size = 128

for category in labels:
    
    folder = os.path.join(dataset_path, category)
    
    for file in os.listdir(folder):
        
        if "mask" in file:
            continue
        
        path = os.path.join(folder, file)
        
        img = cv2.imread(path)
        img = cv2.resize(img, (img_size, img_size))
        img = img / 255.0
        
        X.append(img)
        y.append(labels[category])

X = np.array(X)
y = np.array(y)

print("Dataset shape:", X.shape)

Dataset shape: (780, 128, 128, 3)


In [14]:
unique, counts = np.unique(y, return_counts=True)

for u,c in zip(unique,counts):
    print("Class",u,"count:",c)

Class 0 count: 437
Class 1 count: 210
Class 2 count: 133


## Train Test Split

In [15]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

y_train_cat = to_categorical(y_train, 3)
y_test_cat = to_categorical(y_test, 3)

# CNN Model

In [16]:
def create_cnn():

    model = models.Sequential()

    model.add(layers.Conv2D(32,(3,3),activation='relu',input_shape=(128,128,3)))
    model.add(layers.MaxPooling2D((2,2)))

    model.add(layers.Conv2D(64,(3,3),activation='relu'))
    model.add(layers.MaxPooling2D((2,2)))

    model.add(layers.Conv2D(128,(3,3),activation='relu'))
    model.add(layers.MaxPooling2D((2,2)))

    model.add(layers.Flatten())

    model.add(layers.Dense(128,activation='relu'))
    model.add(layers.Dropout(0.5))

    model.add(layers.Dense(3,activation='softmax'))

    model.compile(
        optimizer='adam',
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    return model

# Baseline CNN

In [17]:
model_baseline = create_cnn()

history_baseline = model_baseline.fit(
    X_train,
    y_train_cat,
    epochs=10,
    batch_size=32,
    validation_data=(X_test,y_test_cat)
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 11s 476ms/step - accuracy: 0.4527 - loss: 1.1499 - val_accuracy: 0.5577 - val_loss: 0.9475
Epoch 2/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 9s 466ms/step - accuracy: 0.5943 - loss: 0.9260 - val_accuracy: 0.5641 - val_loss: 0.8949
Epoch 3/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 9s 448ms/step - accuracy: 0.5914 - loss: 0.8782 - val_accuracy: 0.6154 - val_loss: 0.8723
Epoch 4/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 9s 456ms/step - accuracy: 0.6425 - loss: 0.7831 - val_accuracy: 0.6667 - val_loss: 0.8002
Epoch 5/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 9s 455ms/step - accuracy: 0.6788 - loss: 0.6983 - val_accuracy: 0.6923 - val_loss: 0.7465
Epoch 6/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 9s 453ms/step - accuracy: 0.7443 - loss: 0.6008 - val_accuracy: 0.6859 - val_loss: 0.7298
Epoch 7/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 9s 456ms/step - accuracy: 0.7112 - loss: 0.5976 - val_accuracy: 0.7500 - val_loss: 0.6555
Epoch 8/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 9s 456ms/step - accuracy: 0.7950 - loss: 0.4904 - val_accuracy: 0

## Evaluation 

In [18]:
pred = model_baseline.predict(X_test)

y_pred = np.argmax(pred,axis=1)

print("BASELINE CNN")
print(classification_report(y_test,y_pred))

5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 133ms/step
BASELINE CNN
              precision    recall  f1-score   support

           0       0.71      0.93      0.81        87
           1       0.74      0.62      0.68        42
           2       0.86      0.22      0.35        27

    accuracy                           0.72       156
   macro avg       0.77      0.59      0.61       156
weighted avg       0.74      0.72      0.69       156



# Apply SMOTE

In [19]:
X_train_flat = X_train.reshape(len(X_train), -1)

smote = SMOTE(random_state=42)

X_smote, y_smote = smote.fit_resample(X_train_flat, y_train)

X_smote = X_smote.reshape(-1,128,128,3)

y_smote_cat = to_categorical(y_smote,3)

print("After SMOTE:",np.bincount(y_smote))

After SMOTE: [350 350 350]


## Train Model with SMOTE

In [20]:
model_smote = create_cnn()

history_smote = model_smote.fit(
    X_smote,
    y_smote_cat,
    epochs=10,
    batch_size=32,
    validation_data=(X_test,y_test_cat)
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/10
33/33 ━━━━━━━━━━━━━━━━━━━━ 17s 455ms/step - accuracy: 0.3458 - loss: 1.1608 - val_accuracy: 0.5256 - val_loss: 1.0852
Epoch 2/10
33/33 ━━━━━━━━━━━━━━━━━━━━ 15s 466ms/step - accuracy: 0.4894 - loss: 1.0231 - val_accuracy: 0.5513 - val_loss: 0.9440
Epoch 3/10
33/33 ━━━━━━━━━━━━━━━━━━━━ 15s 441ms/step - accuracy: 0.6589 - loss: 0.7882 - val_accuracy: 0.6538 - val_loss: 0.7784
Epoch 4/10
33/33 ━━━━━━━━━━━━━━━━━━━━ 15s 464ms/step - accuracy: 0.7449 - loss: 0.6221 - val_accuracy: 0.5833 - val_loss: 0.8537
Epoch 5/10
33/33 ━━━━━━━━━━━━━━━━━━━━ 15s 442ms/step - accuracy: 0.7837 - loss: 0.5256 - val_accuracy: 0.7372 - val_loss: 0.7293
Epoch 6/10
33/33 ━━━━━━━━━━━━━━━━━━━━ 15s 447ms/step - accuracy: 0.8751 - loss: 0.3516 - val_accuracy: 0.7308 - val_loss: 0.7247
Epoch 7/10
33/33 ━━━━━━━━━━━━━━━━━━━━ 15s 447ms/step - accuracy: 0.9268 - loss: 0.2282 - val_accuracy: 0.7436 - val_loss: 0.6816
Epoch 8/10
33/33 ━━━━━━━━━━━━━━━━━━━━ 15s 447ms/step - accuracy: 0.9419 - loss: 0.1844 - val_accu

## Evaluate SMOTE Model

In [21]:
pred_smote = model_smote.predict(X_test)

y_pred_smote = np.argmax(pred_smote,axis=1)

print("CNN + SMOTE")
print(classification_report(y_test,y_pred_smote))

5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 138ms/step
CNN + SMOTE
              precision    recall  f1-score   support

           0       0.76      0.90      0.83        87
           1       0.84      0.62      0.71        42
           2       0.70      0.59      0.64        27

    accuracy                           0.77       156
   macro avg       0.77      0.70      0.73       156
weighted avg       0.77      0.77      0.76       156



# Compute Class Weights

In [22]:
weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train
)

class_weights = dict(enumerate(weights))

print(class_weights)

{0: np.float64(0.5942857142857143), 1: np.float64(1.2380952380952381), 2: np.float64(1.9622641509433962)}


## Train Model with Class Weight

In [23]:
model_weight = create_cnn()

history_weight = model_weight.fit(
    X_train,
    y_train_cat,
    epochs=10,
    batch_size=32,
    validation_data=(X_test,y_test_cat),
    class_weight=class_weights
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 11s 474ms/step - accuracy: 0.4185 - loss: 1.1197 - val_accuracy: 0.5962 - val_loss: 1.0393
Epoch 2/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 10s 446ms/step - accuracy: 0.5119 - loss: 1.0051 - val_accuracy: 0.5897 - val_loss: 0.9202
Epoch 3/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 9s 449ms/step - accuracy: 0.6016 - loss: 0.9072 - val_accuracy: 0.6090 - val_loss: 0.8763
Epoch 4/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 9s 452ms/step - accuracy: 0.6431 - loss: 0.8099 - val_accuracy: 0.6859 - val_loss: 0.7551
Epoch 5/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 9s 449ms/step - accuracy: 0.7092 - loss: 0.7208 - val_accuracy: 0.6026 - val_loss: 0.8467
Epoch 6/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 9s 448ms/step - accuracy: 0.6961 - loss: 0.6220 - val_accuracy: 0.6731 - val_loss: 0.7641
Epoch 7/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 9s 447ms/step - accuracy: 0.7556 - loss: 0.5477 - val_accuracy: 0.7179 - val_loss: 0.7065
Epoch 8/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 9s 449ms/step - accuracy: 0.8324 - loss: 0.4267 - val_accuracy: 

## Evaluate Class Weight Model

In [24]:
pred_weight = model_weight.predict(X_test)

y_pred_weight = np.argmax(pred_weight,axis=1)

print("CNN + CLASS WEIGHT")
print(classification_report(y_test,y_pred_weight))

4/5 ━━━━━━━━━━━━━━━━━━━━ 0s 124ms/stepWARNING:tensorflow:5 out of the last 11 calls to <function TensorFlowTrainer.make_predict_function.<locals>.one_step_on_data_distributed at 0x7def7279cd60> triggered tf.function retracing. Tracing is expensive and the excessive number of tracings could be due to (1) creating @tf.function repeatedly in a loop, (2) passing tensors with different shapes, (3) passing Python objects instead of tensors. For (1), please define your @tf.function outside of the loop. For (2), @tf.function has reduce_retracing=True option that can avoid unnecessary retracing. For (3), please refer to https://www.tensorflow.org/guide/function#controlling_retracing and https://www.tensorflow.org/api_docs/python/tf/function for  more details.
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 139ms/step
CNN + CLASS WEIGHT
              precision    recall  f1-score   support

           0       0.83      0.78      0.80        87
           1       0.68      0.64      0.66        42
           2      